# Flight Price Prediction

## Overview

This notebook focuses on building a machine learning model capable of predicting airline ticket prices using historical flight booking information.

The workflow follows a complete machine learning pipeline including data loading, feature engineering, preprocessing, model training, evaluation, cross-validation, and model serialization.

The final trained model will be deployed through Flask REST APIs and later containerized using Docker and Kubernetes.

---

## Objectives

- Load the cleaned dataset from Notebook 2
- Select relevant features
- Perform feature engineering
- Build preprocessing pipeline
- Train multiple regression models
- Compare model performance
- Select the best performing model
- Perform cross validation
- Analyze feature importance
- Serialize the trained model
- Prepare the model for deployment

# Import Required Libraries

The following libraries are used throughout the notebook for data manipulation, visualization, preprocessing, model training, evaluation, and model serialization.

In [ ]:
# Data Manipulation

import pandas as pd
import numpy as np
# Data Manipulation

import pandas as pd
import numpy as np

import sys

try:
    import mlflow
    import mlflow.sklearn
except ModuleNotFoundError:
    !{sys.executable} -m pip install mlflow
    import mlflow
    import mlflow.sklearn

# Visualization

import matplotlib.pyplot as plt
import seaborn as sns

# Ignore warnings

import warnings
warnings.filterwarnings("ignore")

# Model Saving

import joblib
import os
import mlflow.sklearn
# Visualization

import matplotlib.pyplot as plt
import seaborn as sns

# Ignore warnings

import warnings
warnings.filterwarnings("ignore")

# Model Saving

import joblib
import os

ModuleNotFoundError: No module named 'mlflow'

# Machine Learning Libraries

Scikit-Learn and XGBoost are used to build multiple regression models for comparison.

In [ ]:
# Train Test Split

from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)

# Preprocessing

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

# Regression Models

from sklearn.linear_model import LinearRegression

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
    ExtraTreesRegressor
)

from xgboost import XGBRegressor

# Metrics

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Load Preprocessed Dataset

The cleaned and integrated flight dataset generated in Notebook 2 is loaded for model development.

In [ ]:
flight_data = pd.read_csv("../data/processed/flights_preprocessed.csv")

flight_data.head()

,travelCode,userCode,from,to,flightType,price,time,distance,agency,date,year,month,day,day_of_week
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,2019-09-26,2019,9,26,Thursday
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,2019-09-30,2019,9,30,Monday
2,1,0,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,2019-10-03,2019,10,3,Thursday
3,1,0,Florianopolis (SC),Brasilia (DF),firstClass,1127.36,1.66,637.56,CloudFy,2019-10-04,2019,10,4,Friday
4,2,0,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,2019-10-10,2019,10,10,Thursday


# Dataset Overview

Before model development, the dataset is inspected to understand its dimensions, data types, and statistical properties.

In [ ]:
print("Shape :", flight_data.shape)

Shape : (271888, 14)


In [ ]:
flight_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271888 entries, 0 to 271887
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   travelCode   271888 non-null  int64  
 1   userCode     271888 non-null  int64  
 2   from         271888 non-null  object 
 3   to           271888 non-null  object 
 4   flightType   271888 non-null  object 
 5   price        271888 non-null  float64
 6   time         271888 non-null  float64
 7   distance     271888 non-null  float64
 8   agency       271888 non-null  object 
 9   date         271888 non-null  object 
 10  year         271888 non-null  int64  
 11  month        271888 non-null  int64  
 12  day          271888 non-null  int64  
 13  day_of_week  271888 non-null  object 
dtypes: float64(3), int64(5), object(6)
memory usage: 29.0+ MB


In [ ]:
flight_data.describe()

,travelCode,userCode,price,time,distance,year,month,day
count,271888.000000,271888.000000,271888.00000,271888.000000,271888.000000,271888.000000,271888.000000,271888.000000
mean,67971.500000,667.505495,957.37503,1.421147,546.955535,2020.522862,6.607519,15.790458
std,39243.724665,389.523127,362.31189,0.542541,208.851288,0.980161,3.606611,8.826961
min,0.000000,0.000000,301.51000,0.440000,168.220000,2019.000000,1.000000,1.000000
25%,33985.750000,326.000000,672.66000,1.040000,401.660000,2020.000000,3.000000,8.000000
50%,67971.500000,659.000000,904.00000,1.460000,562.140000,2020.000000,7.000000,16.000000
75%,101957.250000,1011.000000,1222.24000,1.760000,676.530000,2021.000000,10.000000,24.000000
max,135943.000000,1339.000000,1754.17000,2.440000,937.770000,2023.000000,12.000000,31.000000


In [ ]:
flight_data.head()

,travelCode,userCode,from,to,flightType,price,time,distance,agency,date,year,month,day,day_of_week
0,0,0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,2019-09-26,2019,9,26,Thursday
1,0,0,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,2019-09-30,2019,9,30,Monday
2,1,0,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,2019-10-03,2019,10,3,Thursday
3,1,0,Florianopolis (SC),Brasilia (DF),firstClass,1127.36,1.66,637.56,CloudFy,2019-10-04,2019,10,4,Friday
4,2,0,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,2019-10-10,2019,10,10,Thursday


# Missing Value Analysis

The dataset is checked for missing values to ensure data completeness before preprocessing.

In [ ]:
flight_data.isnull().sum()

travelCode     0
userCode       0
from           0
to             0
flightType     0
price          0
time           0
distance       0
agency         0
date           0
year           0
month          0
day            0
day_of_week    0
dtype: int64

# Duplicate Record Analysis

Duplicate observations can negatively impact model performance. The dataset is checked for duplicate rows.

In [ ]:
print("Duplicate Rows :", flight_data.duplicated().sum())

Duplicate Rows : 0


# Feature Selection

Only relevant features are retained for model training.

Identifier columns that do not contribute to prediction are removed.

In [ ]:
drop_columns = [
    "travelCode",
    "userCode",
    "hotelName"
]

flight_data.drop(
    columns=drop_columns,
    inplace=True,
    errors="ignore"
)

flight_data.head()

,from,to,flightType,price,time,distance,agency,date,year,month,day,day_of_week
0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,2019-09-26,2019,9,26,Thursday
1,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,2019-09-30,2019,9,30,Monday
2,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,2019-10-03,2019,10,3,Thursday
3,Florianopolis (SC),Brasilia (DF),firstClass,1127.36,1.66,637.56,CloudFy,2019-10-04,2019,10,4,Friday
4,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,2019-10-10,2019,10,10,Thursday


# Feature Engineering

The booking date is decomposed into separate temporal components.

Extracted Features

- Year
- Month
- Day
- Day of Week

In [ ]:
flight_data["date"] = pd.to_datetime(flight_data["date"])

flight_data["year"] = flight_data["date"].dt.year

flight_data["month"] = flight_data["date"].dt.month

flight_data["day"] = flight_data["date"].dt.day

flight_data["day_of_week"] = flight_data["date"].dt.day_name()

flight_data.drop(
    columns=["date"],
    inplace=True
)

flight_data.head()

,from,to,flightType,price,time,distance,agency,year,month,day,day_of_week
0,Recife (PE),Florianopolis (SC),firstClass,1434.38,1.76,676.53,FlyingDrops,2019,9,26,Thursday
1,Florianopolis (SC),Recife (PE),firstClass,1292.29,1.76,676.53,FlyingDrops,2019,9,30,Monday
2,Brasilia (DF),Florianopolis (SC),firstClass,1487.52,1.66,637.56,CloudFy,2019,10,3,Thursday
3,Florianopolis (SC),Brasilia (DF),firstClass,1127.36,1.66,637.56,CloudFy,2019,10,4,Friday
4,Aracaju (SE),Salvador (BH),firstClass,1684.05,2.16,830.86,CloudFy,2019,10,10,Thursday


# Feature Selection and Target Variable

The target variable for this regression problem is the flight ticket price.

The remaining columns are used as input features for model training.

In [ ]:
# Define Features and Target

X = flight_data.drop(columns=["price"])

y = flight_data["price"]

print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Shape : (271888, 10)
Target Shape : (271888,)


# Feature Categorization

The dataset contains both numerical and categorical features.

These feature types require different preprocessing techniques:

- Numerical Features → StandardScaler
- Categorical Features → OneHotEncoder

In [ ]:
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical Features")
print(categorical_features)

print()

print("Numerical Features")
print(numerical_features)

Categorical Features
['from', 'to', 'flightType', 'agency', 'day_of_week']

Numerical Features
['time', 'distance', 'year', 'month', 'day']


# Data Preprocessing Pipeline

A preprocessing pipeline is created using ColumnTransformer.

The pipeline performs:

- Standard Scaling for numerical features.
- One-Hot Encoding for categorical features.

This preprocessing pipeline will later be saved and reused during deployment.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['time', 'distance', 'year', 'month', 'day']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['from', 'to', 'flightType', 'agency',
                                  'day_of_week'])])

In [ ]:
mlflow.set_experiment("TravelOps AI Flight Price")

# Train-Test Split

The dataset is divided into training and testing subsets.

- Training Data : 80%
- Testing Data : 20%

A fixed random state is used to ensure reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Samples :", X_train.shape)

print("Testing Samples :", X_test.shape)

Training Samples : (217510, 10)
Testing Samples : (54378, 10)


# Data Transformation

The preprocessing pipeline is fitted on the training data and then applied to both training and testing datasets.

This prevents data leakage while ensuring consistent preprocessing during inference.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed Training Shape :", X_train_processed.shape)

print("Processed Testing Shape :", X_test_processed.shape)

Processed Training Shape : (217510, 34)
Processed Testing Shape : (54378, 34)


# Verify Processed Data

The transformed datasets are verified to ensure that preprocessing has been successfully applied.

In [ ]:
print(type(X_train_processed))

print(type(X_test_processed))

<class 'scipy.sparse._csr.csr_matrix'>
<class 'scipy.sparse._csr.csr_matrix'>


# Preview Transformed Features

A small sample of the transformed feature matrix is displayed for inspection.

In [ ]:
sample = X_train_processed[:5].toarray()

pd.DataFrame(sample).head()

,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
0,-0.057839,-0.055762,-0.533903,-0.169781,0.477355,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.495122,0.493440,-1.554631,1.216277,-0.542184,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,1.877524,1.870850,-0.533903,0.107431,1.610175,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,-0.573936,-0.579683,-0.533903,-1.278627,-0.202337,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
4,-1.458673,-1.459603,-0.533903,-1.278627,-0.995312,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


# Model Evaluation Function

A reusable evaluation function is created to train and evaluate multiple regression models.

The following evaluation metrics are used:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² Score

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

def evaluate_model(model, X_train, X_test, y_train, y_test):

    # Train Model
    model.fit(X_train, y_train)

    # Prediction
    predictions = model.predict(X_test)

    # Metrics
    mae = mean_absolute_error(y_test, predictions)

    rmse = np.sqrt(
        mean_squared_error(y_test, predictions)
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    return mae, rmse, r2

# Regression Models

Multiple regression algorithms are trained and compared to identify the best-performing model for flight price prediction.

In [ ]:
models = {

    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "AdaBoost": AdaBoostRegressor(
        random_state=42
    ),

    "Extra Trees": ExtraTreesRegressor(
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        random_state=42,
        objective="reg:squarederror"
    )

}

# Model Training

Each regression model is trained using the preprocessed training dataset and evaluated on the testing dataset.

In [ ]:
results = []

for name, model in models.items():

    with mlflow.start_run(run_name=name):

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test, predictions)

        mlflow.log_param("Model", name)

        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2", r2)

        mlflow.sklearn.log_model(model, name)

# Model Performance Comparison

The evaluation metrics of all regression models are compared to identify the most suitable model.

In [ ]:
results_df = pd.DataFrame(

    results,

    columns=[
        "Model",
        "MAE",
        "RMSE",
        "R2 Score"
    ]

)

results_df

,Model,MAE,RMSE,R2 Score
0,Linear Regression,81.314877,103.136318,0.919275
1,Decision Tree,0.008694,0.767013,0.999996
2,Random Forest,0.045863,0.664082,0.999997
3,Gradient Boosting,31.222925,39.222773,0.988325
4,AdaBoost,109.595003,132.786290,0.866189
5,Extra Trees,0.000680,0.028891,1.000000
6,XGBoost,1.042970,1.428163,0.999985


# Ranking Models

Models are ranked based on their R² Score in descending order.

In [ ]:
results_df = results_df.sort_values(
    by="R2 Score",
    ascending=False
)

results_df.reset_index(
    drop=True,
    inplace=True
)

results_df

,Model,MAE,RMSE,R2 Score
0,Extra Trees,0.000680,0.028891,1.000000
1,Random Forest,0.045863,0.664082,0.999997
2,Decision Tree,0.008694,0.767013,0.999996
3,XGBoost,1.042970,1.428163,0.999985
4,Gradient Boosting,31.222925,39.222773,0.988325
5,Linear Regression,81.314877,103.136318,0.919275
6,AdaBoost,109.595003,132.786290,0.866189


# Best Model Selection

The model with the highest R² Score is selected as the final regression model for deployment.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]

print("Best Model :", best_model_name)

Best Model : Extra Trees


In [ ]:
best_model = models[best_model_name]

best_model.fit(
    X_train_processed,
    y_train
)

print("Best model trained successfully.")

Best model trained successfully.


# Model Predictions

The selected regression model is used to predict flight ticket prices on the testing dataset.

In [ ]:
predictions = best_model.predict(
    X_test_processed
)

predictions[:10]

array([ 481.42, 1124.11, 1174.97,  898.67,  959.91, 1367.6 ,  762.89,
       1569.65,  835.21,  674.52])

# Final Model Evaluation

The selected model is evaluated using MAE, RMSE, and R² Score.

In [ ]:
mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

r2 = r2_score(
    y_test,
    predictions
)

print("MAE :", round(mae,2))
print("RMSE :", round(rmse,2))
print("R2 Score :", round(r2,4))

MAE : 0.0
RMSE : 0.03
R2 Score : 1.0


# Prediction Comparison

A comparison between actual and predicted flight prices is presented to evaluate model performance.

In [ ]:
comparison = pd.DataFrame({

    "Actual Price": y_test.values,

    "Predicted Price": predictions

})

comparison.head(10)

,Actual Price,Predicted Price
0,481.42,481.42
1,1124.11,1124.11
2,1174.97,1174.97
3,898.67,898.67
4,959.91,959.91
5,1367.60,1367.60
6,762.89,762.89
7,1569.65,1569.65
8,835.21,835.21
9,674.52,674.52


# Model Serialization

The trained regression model and preprocessing pipeline are serialized using Joblib.

Saving these artifacts enables model deployment without retraining and ensures that the same preprocessing steps are applied during inference.

In [ ]:
import os
import joblib

# Create models directory
os.makedirs("../models", exist_ok=True)

# Save trained model
joblib.dump(
    best_model,
    "../models/flight_price_model.pkl"
)

# Save preprocessing pipeline
joblib.dump(
    preprocessor,
    "../models/flight_preprocessor.pkl"
)

print("Model saved successfully.")
print("Preprocessor saved successfully.")

Model saved successfully.
Preprocessor saved successfully.


# Verify Saved Artifacts

The saved model artifacts are verified to ensure successful serialization.

In [ ]:
import os

print("Saved Files:\n")

for file in os.listdir("../models"):
    print(file)

Saved Files:

flight_preprocessor.pkl
flight_price_model.pkl
gender_label_encoder.pkl
gender_model.pkl
gender_scaler.pkl
recommendation_dataset.pkl
recommendation_model.pkl
recommendation_preprocessor.pkl


# Load Serialized Model

The serialized model and preprocessing pipeline are loaded to verify that they can be reused without retraining.

In [ ]:
loaded_model = joblib.load(
    "../models/flight_price_model.pkl"
)

loaded_preprocessor = joblib.load(
    "../models/flight_preprocessor.pkl"
)

print(type(loaded_model))
print(type(loaded_preprocessor))

<class 'sklearn.ensemble._forest.ExtraTreesRegressor'>
<class 'sklearn.compose._column_transformer.ColumnTransformer'>


# Inference Test

The loaded model is tested on the transformed testing dataset to verify successful serialization and deserialization.

In [ ]:
loaded_predictions = loaded_model.predict(
    X_test_processed
)

loaded_predictions[:10]

array([ 481.42, 1124.11, 1174.97,  898.67,  959.91, 1367.6 ,  762.89,
       1569.65,  835.21,  674.52])

# Prediction Comparison

The first few predicted flight prices are compared against the actual prices.

In [ ]:
comparison = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": loaded_predictions
})

comparison.head(10)

,Actual Price,Predicted Price
0,481.42,481.42
1,1124.11,1124.11
2,1174.97,1174.97
3,898.67,898.67
4,959.91,959.91
5,1367.60,1367.60
6,762.89,762.89
7,1569.65,1569.65
8,835.21,835.21
9,674.52,674.52


# Conclusion

A complete machine learning pipeline for flight price prediction has been successfully developed.

## Workflow Summary

- Loaded the cleaned dataset
- Performed feature engineering
- Built a preprocessing pipeline using ColumnTransformer
- Applied One-Hot Encoding and Standard Scaling
- Split the dataset into training and testing sets
- Trained multiple regression models
- Compared model performance using MAE, RMSE, and R² Score
- Selected the best-performing model
- Serialized the trained model and preprocessing pipeline using Joblib
- Verified the saved artifacts for deployment

## Generated Artifacts

- `flight_price_model.pkl`
- `flight_preprocessor.pkl`

These artifacts are now ready to be integrated into the Flask REST API, followed by MLflow tracking, Docker containerization, Kubernetes deployment, and AWS EC2 hosting as part of the complete TravelOps AI MLOps pipeline.